In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras.optimizers import Adam
import transformers
from datasets import load_dataset
from transformers import AutoTokenizer
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Sequential


In [2]:
dataset=load_dataset("roneneldan/TinyStories",split="train",
                     streaming=True)
dataset=list(dataset.take(20000))

print(dataset[0]["text"])



README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [3]:
print(type(dataset))
print(type(dataset[0]))
text = [sample["text"] for sample in dataset]
Tokenize=TextVectorization(max_tokens=8000,output_mode="int",output_sequence_length=129)
Tokenize.adapt(tf.data.Dataset.from_tensor_slices(text).batch(128))
token_ids=Tokenize(text)
print(token_ids[0][:20])

<class 'list'>
<class 'dict'>
tf.Tensor(
[  26   19    5   30   47   62   17  105    5 1854   13   11  172    8
  151   10    6 1401    4   38], shape=(20,), dtype=int64)


In [4]:

embed_dim=512
no_heads=8
ff_dim=2048

In [21]:
from tensorflow.python.ops.math_ops import xlog1py
vocab_size=len(Tokenize.get_vocabulary())
embed=tf.keras.layers.Embedding(input_dim=vocab_size,output_dim=embed_dim)
x=token_ids[:,:-1]
y=token_ids[:,1:]
train_ds=(tf.data.Dataset.from_tensor_slices((x,y)).shuffle(1000).batch(8).prefetch(1))

class Transformer(Layer):
  def __init__(self,embed_dim,no_heads,ff_dim):
    super().__init__()
    self.att=tf.keras.layers.MultiHeadAttention(num_heads=no_heads,key_dim=embed_dim//no_heads)
    self.ff=tf.keras.Sequential([
        tf.keras.layers.Dense(ff_dim,activation="relu"),
        tf.keras.layers.Dense(embed_dim)
    ])
    self.norm1=tf.keras.layers.LayerNormalization()
    self.norm2=tf.keras.layers.LayerNormalization()
  def call(self,x,training=False):
    res1=self.norm1(x)
    out1=self.att(query=res1,value=res1,use_causal_mask=True,training=training)
    x1=x+out1
    res2=self.norm2(x1)
    out2=self.ff(res2)
    x2=x1+out2
    return x2




In [22]:
class PositionalEmbed(Layer):
  def __init__(self,embed_dim):
    super().__init__()
    position=tf.cast(tf.range(128)[:,tf.newaxis],dtype=tf.float32)
    dim=tf.cast(tf.range(0,embed_dim,2)[tf.newaxis,:],dtype=tf.float32)
    angle_rate=1/tf.pow(10000,dim/embed_dim)
    angle=position*angle_rate
    sin_values=tf.sin(angle)
    cos_value=tf.cos(angle)
    combined=tf.stack([sin_values,cos_value],axis=-1)
    self.pos_encoding=tf.reshape(combined,shape=(128,-1))
  def call(self,x):
    seq_len=tf.shape(x)[1]
    x=x+self.pos_encoding[tf.newaxis,:seq_len,:]
    return x

In [23]:
block = Transformer(embed_dim, no_heads, ff_dim)

for x_batch, y_batch in train_ds.take(1):
    embedded = embed(x_batch)
    output = block(embedded)

    print("Input:", embedded.shape)
    print("Output:", output.shape)

Input: (8, 128, 512)
Output: (8, 128, 512)


In [26]:
class TransformerLM(tf.keras.Model):
    def __init__(self, vocab_size, embed_dim,
                 no_heads, ff_dim, stack=4):

        super().__init__()

        # 1. Token embedding layer
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_dim)


        self.position = PositionalEmbed(embed_dim)

        # 3. Stack independent Transformer blocks
        self.blocks = [
            Transformer(embed_dim, no_heads, ff_dim)
            for _ in range(stack)
        ]

        # 4. Final normalization
        self.final_norm = tf.keras.layers.LayerNormalization()

        # 5. Vocabulary projection
        self.lm_head = tf.keras.layers.Dense(vocab_size)
    def call(self, token_ids, training=False):


      x = self.embedding(token_ids)


      x = self.position(x)


      for block in self.blocks:
        x = block(x, training=training)


      x = self.final_norm(x)


      logits = self.lm_head(x)

      return logits

In [27]:
model = TransformerLM(
    vocab_size, embed_dim, no_heads, ff_dim, stack=4
)

for x_batch, y_batch in train_ds.take(1):
    logits = model(x_batch)

    print("Input:", x_batch.shape)
    print("Output:", logits.shape)
    print("Targets:", y_batch.shape)

Input: (8, 128)
Output: (8, 128, 8000)
Targets: (8, 128)
